# Notebook: 01-03-Basics - Building a Conversational AI System with Memory Management in DynamoDB
___
## Overview
This notebook demonstrates the setup and implementation of a conversational AI system using AWS services and MLflow for tracking. The system is designed to handle customer interactions with intelligence and maintain conversation state.




# 1.Initial setup
___
### Essential imports for the conversational AI system
- langgraph: For building conversational flows
- psycopg2: For database interactions
- MLflow: For experiment tracking and monitoring
- boto3: For AWS service interactions

In [1]:
# %%capture --no-stderr
# !pip install -U langgraph
# !pip install psycopg2-binary==2.9.9
# !pip install python-dotenv

In [2]:
from pprint import pprint

In [3]:
import os
import sys
import boto3
import json
import logging
import mlflow
from mlflow.entities import SpanType


from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [4]:
mlflow.__version__

'2.20.3'

# Setup Path and Environment
Add backend directory to system path for module imports

In [5]:
# Get the absolute path to the backend directory
current_dir = os.getcwd()  # Get current working directory
backend_path = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(backend_path)

# Load environment variables from .env file
load_dotenv()

True

## Importing Custom libraries
___
Now we are using the app library that is basically handling the state management in our behalf.  
This is an abstraction layer to handle the retreival and persist operation into dynamoDB os the latest state of the interaction

In [6]:
from backend import app

# 2. MLflow Configuration
___
- Sets up MLflow tracking with AWS SageMaker
- Enables auto-logging for LangChain and/or Bedrock interactions
- Configures MLflow tracking URI using environment variables


In [7]:
# # Enable auto-tracing for Amazon Bedrock
mlflow.bedrock.autolog()
mlflow.langchain.autolog()

2025/03/25 17:57:11 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.


In [8]:
try:
    mlflow_arn = os.getenv('MLFLOW_TRACKING_ARN')
    if mlflow_arn:
        print(f'\nMLFlow tracking URI: {mlflow_arn}')
        mlflow.set_tracking_uri(mlflow_arn)
        print('Successfully connected to MLFlow tracking server')
except Exception as e:
    print('Error connecting to MLFlow tracking server')
    raise e


MLFlow tracking URI: arn:aws:sagemaker:us-west-2:577201992296:mlflow-tracking-server/bedrock-chatbot-mlflow
Successfully connected to MLFlow tracking server


In [9]:
import uuid
session_id = str(uuid.uuid4())

In [10]:
session_id

'ad49ebdb-d9fb-4dba-a6df-4f653c958400'

In [11]:
mlflow.set_experiment(str(session_id))
mlflow.tracing.enable_notebook_display()

2025/03/25 17:57:13 INFO mlflow.tracking.fluent: Experiment with name 'ad49ebdb-d9fb-4dba-a6df-4f653c958400' does not exist. Creating a new experiment.


In [ ]:
# Initialize test message
messages = 'I need help with an order.'
# Alternative test message with more details:
# messages = "I need help with an order. I don't remember my order id. My email is john@example.com"


In [ ]:
# Create initial session dictionary
session_dict = {'session_id': session_id}

In [ ]:
# Execute the chatbot workflow
# This processes the user's message through the conversation flow
response = app.execute_workflow(
    task=messages,             # The user's input message
    session_dict=session_dict  # Session context containing unique identifier
)

# Extract and display the chatbot's response
# Gets the 'response' field from the workflow execution result
response.get('response')


 <============== Node: entry_intent ===================> 



"<reply>Hello! I'd be happy to help you with your order. To assist you better, I'll need either:\n1. Your order ID number, or\n2. Your email address, phone number, or username associated with your account\n\nCould you please provide one of these details?</reply>"

In [15]:
messages = "My order number is 13579"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <============== Node: entry_intent ===================> 


 <=============== Node: order_confirmation =============> 



"<reply>I see Order #13579 in our system. Could you please confirm if this is the order you'd like me to look up for you?</reply>"

In [16]:
messages = "Perfect that's the order"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <=============== Node: order_confirmation =============> 


 <=============== Node: resolution =====================> 


 ------- RETRY 1 with delay 1.05s ---------- 



'<reply>Thank you for confirming. Since your order is currently in processing status, I can help you with canceling or updating this order if needed. What would you like to do with this order?</reply>'

In [17]:
messages = "I want to cancel the order"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <=============== Node: resolution =====================> 


 ------- RETRY 1 with delay 1.05s ---------- 


 ------- CANCEL ORDER: 13579 ---------- 


 ------- QUERY: UPDATE orders SET status = 'Cancelled' WHERE id = %s AND status = 'Processing' ----------- 

Cancel successful: Mockup function not real implementation

 ------- RETRY 1 with delay 1.01s ---------- 



"<reply>I've successfully cancelled your order #13579. You should receive a confirmation email shortly. Is there anything else I can help you with?</reply>"

In [18]:
messages = "I'm all set. Thank you"
response = app.execute_workflow(
    task=messages,
    session_dict=session_dict
)
response.get('response')


 <=============== Node: resolution =====================> 



"<thinking>\nThe user has indicated they don't need any further assistance and has expressed gratitude. I should provide a polite closing response.\n</thinking>\n\n<reply>You're welcome! Thank you for contacting AnyCompany customer service. Have a great day!</reply>"

### Notes:
- The main difference between the notebooks 01-02 and 01-03 are:
    - We are not building the graph in the notebook we are building on a run-time the graph through the app lib.
    - Eventhough the Nodes and Graph code are the same we now are using from the main library, mimic real application.
    - The most important feature to highlight is that now the memory management is done through the app library and we are interacting with DynamoDB.